# LBO Modeling From Entry To Exit

This notebook is intentionally written as a beginner-friendly teaching model. The goal is not to write compact production code. The goal is to make the financial logic visible.

The workflow is:

1. Start with simple editable assumptions.
2. Calculate entry valuation.
3. Build Sources & Uses.
4. Project operating performance.
5. Sweep cash into debt repayment.
6. Calculate lender metrics.
7. Calculate sponsor returns.
8. Run a small sensitivity analysis.
9. Reconcile the notebook against the Streamlit engine.


## 0. Setup

This cell imports the packages used in the notebook. The financial model starts in the next section.


In [1]:
# sys and pathlib help the notebook find the local project modules.
import sys
from pathlib import Path
from copy import deepcopy

# numpy is used for sensitivity grids and arrays.
import numpy as np

# pandas is used for readable finance tables.
import pandas as pd

# plotly express is the simplest Plotly interface for clear interactive charts.
import plotly.express as px

# graph_objects is used only when a slightly more custom chart is useful.
import plotly.graph_objects as go

# IPython display makes notebook outputs cleaner.
from IPython.display import Markdown, display

# numpy_financial gives us the IRR function used in finance.
import numpy_financial as npf

# This makes imports work from either the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()

# Add the project root to Python's import path if it is not already there.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# The engine is used only at the end to check consistency with the Streamlit demo.
from utils.lbo_engine import run_lbo_model

# The optional reporting layer is used at the end.
from utils.gemini_reporting import build_commentary_payload, generate_investment_commentary

# Make pandas print fewer decimals by default.
pd.options.display.float_format = "{:,.2f}".format

# A small color set keeps charts consistent without complex chart styling.
color_map = {
    "blue": "#1f77b4",
    "green": "#2ca02c",
    "red": "#d62728",
    "orange": "#ff7f0e",
    "gray": "#7f7f7f",
}

print("Setup complete.")
print(f"Project root: {PROJECT_ROOT}")


Setup complete.
Project root: C:\Users\julio\Documents\DemoCode\FinEngineering_AlgoTrading


## 1. Editable Base Case Assumptions

The assumptions are deliberately stored as simple Python variables. This is easier for beginners than starting with nested dictionaries or custom classes.


In [2]:
# -----------------------------
# Deal inputs
# -----------------------------

# Current share price before the acquisition offer.
share_price = 2.50

# Fully diluted shares outstanding, in millions.
diluted_shares = 369.0

# Net debt in EUR millions.
net_debt = 135.2


# -----------------------------
# Valuation assumptions
# -----------------------------

# Entry multiple used to buy the business.
entry_multiple = 7.0


# -----------------------------
# Financing assumptions
# -----------------------------

# Senior debt as a percentage of total funding.
senior_debt_pct = 0.35

# Subordinated debt as a percentage of total funding.
subordinated_debt_pct = 0.35

# Sponsor equity is whatever is left after the debt financing.
sponsor_equity_pct = 1 - senior_debt_pct - subordinated_debt_pct

# Annual interest rate on senior debt.
senior_interest_rate = 0.06

# Annual interest rate on subordinated debt.
subordinated_interest_rate = 0.065


# -----------------------------
# Historical operating inputs
# -----------------------------

# EBIT is operating profit before interest and tax.
historical_ebit = 163.9

# D&A is depreciation and amortization, a non-cash accounting expense.
historical_d_and_a = 76.9

# Operating working capital balance.
historical_owc = -86.9

# Capital expenditure.
historical_capex = 127.7


# -----------------------------
# Tax, fees, and exit assumptions
# -----------------------------

# Transaction fees as a percentage of entry enterprise value.
fees_pct_of_ev = 0.005

# Corporate tax rate.
tax_rate = 0.35

# Exit multiple used when the sponsor sells the business.
exit_multiple = 7.0

# Exit year.
exit_year = 3


# Put the inputs into a simple table for display.
input_table = pd.DataFrame(
    [
        ["Share price", f"EUR {share_price:.2f}"],
        ["Diluted shares", f"{diluted_shares:.1f}m"],
        ["Net debt", f"EUR {net_debt:.1f}m"],
        ["Entry multiple", f"{entry_multiple:.1f}x"],
        ["Senior debt %", f"{senior_debt_pct:.1%}"],
        ["Subordinated debt %", f"{subordinated_debt_pct:.1%}"],
        ["Sponsor equity %", f"{sponsor_equity_pct:.1%}"],
        ["Senior interest rate", f"{senior_interest_rate:.1%}"],
        ["Subordinated interest rate", f"{subordinated_interest_rate:.1%}"],
        ["Historical EBIT", f"EUR {historical_ebit:.1f}m"],
        ["Historical D&A", f"EUR {historical_d_and_a:.1f}m"],
        ["Historical OWC", f"EUR {historical_owc:.1f}m"],
        ["Historical capex", f"EUR {historical_capex:.1f}m"],
        ["Fees as % of EV", f"{fees_pct_of_ev:.1%}"],
        ["Tax rate", f"{tax_rate:.1%}"],
        ["Exit multiple", f"{exit_multiple:.1f}x"],
        ["Exit year", f"Year {exit_year}"],
    ],
    columns=["Assumption", "Value"],
)

display(input_table)

# Show the financing mix visually.
financing_mix = pd.DataFrame(
    {
        "Funding source": ["Senior debt", "Subordinated debt", "Sponsor equity"],
        "Share": [senior_debt_pct, subordinated_debt_pct, sponsor_equity_pct],
    }
)

fig = px.bar(
    financing_mix,
    x="Funding source",
    y="Share",
    text=financing_mix["Share"].map(lambda x: f"{x:.0%}"),
    title="Base Case Financing Mix",
    color="Funding source",
)
fig.update_yaxes(tickformat=".0%", title="Share of total funding")
fig.update_traces(textposition="outside")
fig.show()


,Assumption,Value
0,Share price,EUR 2.50
1,Diluted shares,369.0m
2,Net debt,EUR 135.2m
3,Entry multiple,7.0x
4,Senior debt %,35.0%
5,Subordinated debt %,35.0%
6,Sponsor equity %,30.0%
7,Senior interest rate,6.0%
8,Subordinated interest rate,6.5%
9,Historical EBIT,EUR 163.9m


## 2. Entry Valuation

This section turns historical operating performance into the purchase price.


In [3]:
# EBITDA is EBIT plus D&A.
# We add back D&A because it is a non-cash accounting expense.
historical_ebitda = historical_ebit + historical_d_and_a

# Enterprise Value is the value of the whole business.
# In this case it is calculated as EBITDA times the entry multiple.
entry_enterprise_value = historical_ebitda * entry_multiple

# Equity Value is what shareholders receive after net debt is deducted.
implied_equity_value = entry_enterprise_value - net_debt

# Offer price per share divides total equity value by diluted shares.
implied_offer_price = implied_equity_value / diluted_shares

# Takeover premium compares the implied offer price with the current share price.
takeover_premium = implied_offer_price / share_price - 1

# Build a table so every step is visible.
entry_valuation = pd.DataFrame(
    [
        ["Historical EBIT", historical_ebit, "Starting operating profit"],
        ["D&A", historical_d_and_a, "Added back because it is non-cash"],
        ["Historical EBITDA", historical_ebitda, "EBIT + D&A"],
        ["Entry Enterprise Value", entry_enterprise_value, "EBITDA x entry multiple"],
        ["Less: Net debt", -net_debt, "Debt reduces value available to shareholders"],
        ["Implied Equity Value", implied_equity_value, "Enterprise value - net debt"],
        ["Implied offer price per share", implied_offer_price, "Equity value / diluted shares"],
        ["Takeover premium", takeover_premium, "Offer price / current share price - 1"],
    ],
    columns=["Metric", "Value", "Explanation"],
)

entry_valuation_display = entry_valuation.copy()

# Format the values for human reading.
entry_valuation_display["Value"] = [
    f"EUR {historical_ebit:.1f}m",
    f"EUR {historical_d_and_a:.1f}m",
    f"EUR {historical_ebitda:.1f}m",
    f"EUR {entry_enterprise_value:.1f}m",
    f"EUR {-net_debt:.1f}m",
    f"EUR {implied_equity_value:.1f}m",
    f"EUR {implied_offer_price:.2f}",
    f"{takeover_premium:.1%}",
]

display(entry_valuation_display)

# Simple bridge chart: enterprise value, net debt, and equity value.
valuation_chart = pd.DataFrame(
    {
        "Step": ["Enterprise Value", "Net Debt", "Equity Value"],
        "EURm": [entry_enterprise_value, -net_debt, implied_equity_value],
    }
)

fig = px.bar(
    valuation_chart,
    x="Step",
    y="EURm",
    text=valuation_chart["EURm"].map(lambda x: f"{x:,.1f}"),
    title="Entry Valuation Bridge",
)
fig.update_yaxes(title="EUR millions")
fig.update_traces(textposition="outside")
fig.show()


,Metric,Value,Explanation
0,Historical EBIT,EUR 163.9m,Starting operating profit
1,D&A,EUR 76.9m,Added back because it is non-cash
2,Historical EBITDA,EUR 240.8m,EBIT + D&A
3,Entry Enterprise Value,EUR 1685.6m,EBITDA x entry multiple
4,Less: Net debt,EUR -135.2m,Debt reduces value available to shareholders
5,Implied Equity Value,EUR 1550.4m,Enterprise value - net debt
6,Implied offer price per share,EUR 4.20,Equity value / diluted shares
7,Takeover premium,68.1%,Offer price / current share price - 1


## 3. Sources & Uses

Sources & Uses is the closing funding schedule. Uses show what must be paid. Sources show where the money comes from.


In [4]:
# The sponsor must buy the equity value calculated above.
use_acquisition_of_equity = implied_equity_value

# The transaction also refinances or assumes net debt.
use_refinanced_debt = net_debt

# Fees are calculated as a percentage of enterprise value.
use_fees = entry_enterprise_value * fees_pct_of_ev

# Total uses are the total cash needs at closing.
total_uses = use_acquisition_of_equity + use_refinanced_debt + use_fees

# Senior debt is a percentage of total uses.
source_senior_debt = total_uses * senior_debt_pct

# Subordinated debt is also a percentage of total uses.
source_subordinated_debt = total_uses * subordinated_debt_pct

# Sponsor equity is the residual funding need.
source_sponsor_equity = total_uses - source_senior_debt - source_subordinated_debt

# Total sources should equal total uses.
total_sources = source_senior_debt + source_subordinated_debt + source_sponsor_equity

# Keep this variable name for later sections.
sponsor_equity = source_sponsor_equity

sources_and_uses = pd.DataFrame(
    {
        "Uses": [
            "Acquisition of equity",
            "Refinanced debt",
            "Fees",
            "Total Uses",
        ],
        "Uses EURm": [
            use_acquisition_of_equity,
            use_refinanced_debt,
            use_fees,
            total_uses,
        ],
        "Sources": [
            "Senior debt",
            "Subordinated debt",
            "Sponsor equity",
            "Total Sources",
        ],
        "Sources EURm": [
            source_senior_debt,
            source_subordinated_debt,
            source_sponsor_equity,
            total_sources,
        ],
    }
)

display(sources_and_uses)

print(f"Check: Total Uses - Total Sources = EUR {total_uses - total_sources:.6f}m")

# Plot uses and sources in one simple grouped chart.
funding_chart = pd.DataFrame(
    {
        "Item": [
            "Acquisition of equity",
            "Refinanced debt",
            "Fees",
            "Senior debt",
            "Subordinated debt",
            "Sponsor equity",
        ],
        "EURm": [
            use_acquisition_of_equity,
            use_refinanced_debt,
            use_fees,
            source_senior_debt,
            source_subordinated_debt,
            source_sponsor_equity,
        ],
        "Side": ["Uses", "Uses", "Uses", "Sources", "Sources", "Sources"],
    }
)

fig = px.bar(
    funding_chart,
    x="Item",
    y="EURm",
    color="Side",
    text=funding_chart["EURm"].map(lambda x: f"{x:,.1f}"),
    title="Sources & Uses",
)
fig.update_yaxes(title="EUR millions")
fig.update_xaxes(tickangle=-25)
fig.update_traces(textposition="outside")
fig.show()


,Uses,Uses EURm,Sources,Sources EURm
0,Acquisition of equity,"1,550.40",Senior debt,592.91
1,Refinanced debt,135.20,Subordinated debt,592.91
2,Fees,8.43,Sponsor equity,508.21
3,Total Uses,"1,694.03",Total Sources,"1,694.03"


Check: Total Uses - Total Sources = EUR 0.000000m


## 4. Operating Projection

The transaction is now closed. The next question is whether the business generates enough cash to repay debt.


In [5]:
# Build the operating projection directly as a small table.
# Year 0 is historical. Years 1-3 match the classroom Excel case.
operating_projection = pd.DataFrame(
    {
        "Year": ["Historical", "Year 1", "Year 2", "Year 3"],
        "EBIT": [163.9, 177.2925, 186.157125, 195.46498125],
        "D&A": [76.9, 81.8978, 84.7074766, 87.7720210226],
        "OWC": [-86.9, -92.1921, -96.801705, -101.64179025],
        "Capex": [127.7, 113.4672, 119.14056, 125.097588],
    }
)

# EBITDA is calculated, not manually typed.
operating_projection["EBITDA"] = operating_projection["EBIT"] + operating_projection["D&A"]

# Change in OWC is prior year OWC minus current year OWC.
# In this case, OWC is negative, so becoming more negative releases cash.
operating_projection["Change in OWC"] = operating_projection["OWC"].shift(1) - operating_projection["OWC"]

# Historical year has no prior period inside the model.
operating_projection["Change in OWC"] = operating_projection["Change in OWC"].fillna(0.0)

display(operating_projection)

# Chart EBIT and EBITDA.
profit_chart = operating_projection[operating_projection["Year"] != "Historical"]

fig = px.line(
    profit_chart,
    x="Year",
    y=["EBIT", "EBITDA"],
    markers=True,
    title="Operating Profit Projection",
)
fig.update_yaxes(title="EUR millions")
fig.show()

# Chart capex and working-capital movement.
cash_items = profit_chart[["Year", "Capex", "Change in OWC"]].melt(id_vars="Year", var_name="Cash item", value_name="EURm")

fig = px.bar(
    cash_items,
    x="Year",
    y="EURm",
    color="Cash item",
    barmode="group",
    title="Capex And Working Capital Movement",
)
fig.update_yaxes(title="EUR millions")
fig.show()


,Year,EBIT,D&A,OWC,Capex,EBITDA,Change in OWC
0,Historical,163.90,76.90,-86.90,127.70,240.80,0.00
1,Year 1,177.29,81.90,-92.19,113.47,259.19,5.29
2,Year 2,186.16,84.71,-96.80,119.14,270.86,4.61
3,Year 3,195.46,87.77,-101.64,125.10,283.24,4.84


## 5. Year 1 Cash Flow Available For Debt Repayment

This cell shows the cash-flow logic for one year before we automate it for the full debt schedule.


In [6]:
# Select Year 1 from the operating projection.
year_1 = operating_projection.loc[operating_projection["Year"] == "Year 1"].iloc[0]

# Beginning debt comes from Sources & Uses.
beginning_senior_debt = source_senior_debt
beginning_subordinated_debt = source_subordinated_debt

# Start with no repayment estimate.
senior_repayment = 0.0
subordinated_repayment = 0.0

# Store each iteration so students can see the circularity converge.
iteration_rows = []

# We iterate because repayment affects average debt, average debt affects interest,
# and interest affects cash available for repayment.
for iteration in range(1, 21):
    # Interest is charged on average debt during the year.
    senior_interest = senior_interest_rate * max(beginning_senior_debt - 0.5 * senior_repayment, 0.0)
    subordinated_interest = subordinated_interest_rate * max(beginning_subordinated_debt - 0.5 * subordinated_repayment, 0.0)

    # Total interest is the cash cost of debt.
    total_interest = senior_interest + subordinated_interest

    # Profit before tax is EBIT after interest expense.
    profit_before_tax = year_1["EBIT"] - total_interest

    # Taxes are based on profit before tax.
    taxes = profit_before_tax * tax_rate

    # This is the core LBO cash-flow identity.
    cash_available_for_debt_repayment = (
        year_1["EBIT"]
        + year_1["D&A"]
        - total_interest
        - taxes
        + year_1["Change in OWC"]
        - year_1["Capex"]
    )

    # Senior debt is repaid first.
    new_senior_repayment = min(beginning_senior_debt, max(cash_available_for_debt_repayment, 0.0))

    # Subordinated debt receives only the leftover cash after senior debt.
    new_subordinated_repayment = min(
        beginning_subordinated_debt,
        max(cash_available_for_debt_repayment - new_senior_repayment, 0.0),
    )

    # Save the iteration result for display.
    iteration_rows.append(
        [
            iteration,
            senior_interest,
            subordinated_interest,
            cash_available_for_debt_repayment,
            new_senior_repayment,
            new_subordinated_repayment,
        ]
    )

    # Stop if the estimate no longer changes.
    if abs(new_senior_repayment - senior_repayment) < 1e-10 and abs(new_subordinated_repayment - subordinated_repayment) < 1e-10:
        senior_repayment = new_senior_repayment
        subordinated_repayment = new_subordinated_repayment
        break

    # Update the repayment estimate and repeat.
    senior_repayment = new_senior_repayment
    subordinated_repayment = new_subordinated_repayment

iteration_table = pd.DataFrame(
    iteration_rows,
    columns=[
        "Iteration",
        "Senior interest",
        "Subordinated interest",
        "Cash available",
        "Senior repayment",
        "Subordinated repayment",
    ],
)

display(iteration_table)

# Show the Year 1 cash bridge.
cash_bridge = pd.DataFrame(
    {
        "Cash flow item": ["EBIT", "D&A", "Interest", "Taxes", "Change in OWC", "Capex", "Cash available"],
        "EURm": [
            year_1["EBIT"],
            year_1["D&A"],
            -total_interest,
            -taxes,
            year_1["Change in OWC"],
            -year_1["Capex"],
            cash_available_for_debt_repayment,
        ],
    }
)

display(cash_bridge)

fig = px.bar(
    cash_bridge,
    x="Cash flow item",
    y="EURm",
    text=cash_bridge["EURm"].map(lambda x: f"{x:,.1f}"),
    title="Year 1 Cash Flow Available For Debt Repayment",
)
fig.update_yaxes(title="EUR millions")
fig.update_xaxes(tickangle=-25)
fig.update_traces(textposition="outside")
fig.show()


,Iteration,Senior interest,Subordinated interest,Cash available,Senior repayment,Subordinated repayment
0,1,35.57,38.54,40.79,40.79,0.00
1,2,34.35,38.54,41.58,41.58,0.00
2,3,34.33,38.54,41.60,41.60,0.00
3,4,34.33,38.54,41.60,41.60,0.00
4,5,34.33,38.54,41.60,41.60,0.00
5,6,34.33,38.54,41.60,41.60,0.00
6,7,34.33,38.54,41.60,41.60,0.00
7,8,34.33,38.54,41.60,41.60,0.00


,Cash flow item,EURm
0,EBIT,177.29
1,D&A,81.90
2,Interest,-72.87
3,Taxes,-36.55
4,Change in OWC,5.29
5,Capex,-113.47
6,Cash available,41.60


## 6. Full Debt Schedule

Now we repeat the same cash sweep for Years 1, 2, and 3.


In [7]:
# Reset beginning debt balances to the closing debt amounts.
beginning_senior_debt = source_senior_debt
beginning_subordinated_debt = source_subordinated_debt

# The model starts with zero cash balance.
beginning_cash_balance = 0.0

# This list will collect one row per projected year.
debt_rows = []

# Loop through each projected year.
for year_label in ["Year 1", "Year 2", "Year 3"]:
    # Pull the operating row for the current year.
    row = operating_projection.loc[operating_projection["Year"] == year_label].iloc[0]

    # Start each year with zero repayment estimate.
    senior_repayment = 0.0
    subordinated_repayment = 0.0

    # Iterate to solve the interest / repayment circularity.
    for iteration in range(1, 51):
        senior_interest = senior_interest_rate * max(beginning_senior_debt - 0.5 * senior_repayment, 0.0)
        subordinated_interest = subordinated_interest_rate * max(beginning_subordinated_debt - 0.5 * subordinated_repayment, 0.0)
        total_interest = senior_interest + subordinated_interest
        profit_before_tax = row["EBIT"] - total_interest
        taxes = profit_before_tax * tax_rate

        cash_available = (
            row["EBIT"]
            + row["D&A"]
            - total_interest
            - taxes
            + row["Change in OWC"]
            - row["Capex"]
        )

        cash_for_sweep = max(beginning_cash_balance + cash_available, 0.0)
        new_senior_repayment = min(beginning_senior_debt, cash_for_sweep)
        new_subordinated_repayment = min(beginning_subordinated_debt, max(cash_for_sweep - new_senior_repayment, 0.0))

        if abs(new_senior_repayment - senior_repayment) < 1e-10 and abs(new_subordinated_repayment - subordinated_repayment) < 1e-10:
            senior_repayment = new_senior_repayment
            subordinated_repayment = new_subordinated_repayment
            break

        senior_repayment = new_senior_repayment
        subordinated_repayment = new_subordinated_repayment

    # Ending debt is beginning debt minus repayment.
    ending_senior_debt = max(beginning_senior_debt - senior_repayment, 0.0)
    ending_subordinated_debt = max(beginning_subordinated_debt - subordinated_repayment, 0.0)

    # Ending cash is beginning cash plus cash available minus debt repayment.
    ending_cash_balance = beginning_cash_balance + cash_available - senior_repayment - subordinated_repayment

    # Save the year in the debt schedule.
    debt_rows.append(
        {
            "Year": year_label,
            "Beginning senior debt": beginning_senior_debt,
            "Beginning subordinated debt": beginning_subordinated_debt,
            "Interest expense on senior debt": senior_interest,
            "Interest expense on subordinated debt": subordinated_interest,
            "Interest expense": total_interest,
            "Profit before tax": profit_before_tax,
            "Tax expense": taxes,
            "Cash flow available for debt repayment": cash_available,
            "Repayment of senior debt": senior_repayment,
            "Repayment of subordinated debt": subordinated_repayment,
            "Ending senior debt": ending_senior_debt,
            "Ending subordinated debt": ending_subordinated_debt,
            "Ending cash balance": ending_cash_balance,
        }
    )

    # Ending balances become next year's beginning balances.
    beginning_senior_debt = ending_senior_debt
    beginning_subordinated_debt = ending_subordinated_debt
    beginning_cash_balance = ending_cash_balance

debt_schedule = pd.DataFrame(debt_rows)

display(debt_schedule)

# Total ending debt is senior debt plus subordinated debt.
debt_schedule["Ending total debt"] = debt_schedule["Ending senior debt"] + debt_schedule["Ending subordinated debt"]

fig = px.bar(
    debt_schedule,
    x="Year",
    y=["Ending senior debt", "Ending subordinated debt"],
    title="Ending Debt Balance By Year",
)
fig.update_yaxes(title="EUR millions")
fig.show()

fig = px.line(
    debt_schedule,
    x="Year",
    y="Ending total debt",
    markers=True,
    title="Total Debt Paydown",
)
fig.update_yaxes(title="EUR millions")
fig.show()


,Year,Beginning senior debt,Beginning subordinated debt,Interest expense on senior debt,Interest expense on subordinated debt,Interest expense,Profit before tax,Tax expense,Cash flow available for debt repayment,Repayment of senior debt,Repayment of subordinated debt,Ending senior debt,Ending subordinated debt,Ending cash balance
0,Year 1,592.91,592.91,34.33,38.54,72.87,104.43,36.55,41.60,41.60,0.00,551.31,592.91,0.00
1,Year 2,551.31,592.91,31.71,38.54,70.25,115.90,40.57,45.51,45.51,0.00,505.80,592.91,0.00
2,Year 3,505.80,592.91,28.82,38.54,67.36,128.10,44.84,50.78,50.78,0.00,455.01,592.91,0.00


## 7. Credit Metrics

Lenders usually focus on leverage and interest coverage.


In [8]:
# Keep only projected operating years.
projected_operating = operating_projection[operating_projection["Year"] != "Historical"].copy()

# Total debt is ending senior debt plus ending subordinated debt.
ending_total_debt = debt_schedule["Ending senior debt"] + debt_schedule["Ending subordinated debt"]

# Interest coverage tells us how many times EBITDA covers interest expense.
ebitda_interest_coverage = projected_operating["EBITDA"].to_numpy() / debt_schedule["Interest expense"].to_numpy()

# Debt / EBITDA tells us how levered the business remains.
total_debt_to_ebitda = ending_total_debt.to_numpy() / projected_operating["EBITDA"].to_numpy()

# Total debt repaid shows cumulative deleveraging.
opening_total_debt = source_senior_debt + source_subordinated_debt
percent_total_debt_repaid = 1 - ending_total_debt.to_numpy() / opening_total_debt

# Senior debt repaid focuses on the safest debt layer.
percent_senior_debt_repaid = 1 - debt_schedule["Ending senior debt"].to_numpy() / debt_schedule["Beginning senior debt"].to_numpy()

credit_metrics = pd.DataFrame(
    {
        "Year": debt_schedule["Year"],
        "EBITDA / Interest expense": ebitda_interest_coverage,
        "Total Debt / EBITDA": total_debt_to_ebitda,
        "% of total debt repaid": percent_total_debt_repaid,
        "% of senior debt repaid": percent_senior_debt_repaid,
    }
)

display(credit_metrics)

fig = px.line(
    credit_metrics,
    x="Year",
    y=["Total Debt / EBITDA", "EBITDA / Interest expense"],
    markers=True,
    title="Lender Metrics",
)
fig.update_yaxes(title="x")
fig.show()


,Year,EBITDA / Interest expense,Total Debt / EBITDA,% of total debt repaid,% of senior debt repaid
0,Year 1,3.56,4.41,0.04,0.07
1,Year 2,3.86,4.06,0.07,0.08
2,Year 3,4.20,3.70,0.12,0.10


## 8. Exit Returns

At exit, the sponsor sells the company, repays remaining net debt, and receives the remaining equity value.


In [9]:
# Select the exit operating year.
exit_operating_row = operating_projection.loc[operating_projection["Year"] == f"Year {exit_year}"].iloc[0]

# Exit EBITDA is the EBITDA in the exit year.
exit_ebitda = exit_operating_row["EBITDA"]

# Exit enterprise value equals exit EBITDA times the exit multiple.
exit_enterprise_value = exit_ebitda * exit_multiple

# Select the debt schedule row for the exit year.
exit_debt_row = debt_schedule.loc[debt_schedule["Year"] == f"Year {exit_year}"].iloc[0]

# Exit net debt is remaining debt minus cash.
exit_net_debt = exit_debt_row["Ending senior debt"] + exit_debt_row["Ending subordinated debt"] - exit_debt_row["Ending cash balance"]

# Exit equity value is what the sponsor receives after debt is repaid.
exit_equity_value = exit_enterprise_value - exit_net_debt

# Sponsor cash flows: money out at entry, no dividends, money back at exit.
sponsor_cash_flows = [-sponsor_equity] + [0.0] * (exit_year - 1) + [exit_equity_value]

# IRR is the annualized return on those cash flows.
sponsor_irr = float(npf.irr(sponsor_cash_flows))

# MOIC is total cash received divided by cash invested.
sponsor_moic = exit_equity_value / sponsor_equity

returns_summary = pd.DataFrame(
    {
        "Metric": [
            "Exit EBITDA",
            "Exit Enterprise Value",
            "Exit net debt",
            "Exit Equity Value",
            "Sponsor equity invested",
            "IRR",
            "MOIC",
        ],
        "Value": [
            exit_ebitda,
            exit_enterprise_value,
            exit_net_debt,
            exit_equity_value,
            sponsor_equity,
            sponsor_irr,
            sponsor_moic,
        ],
    }
)

display(returns_summary)

cash_flow_table = pd.DataFrame(
    {
        "Year": [f"Year {i}" for i in range(len(sponsor_cash_flows))],
        "Sponsor cash flow": sponsor_cash_flows,
    }
)

display(cash_flow_table)

fig = px.bar(
    cash_flow_table,
    x="Year",
    y="Sponsor cash flow",
    title=f"Sponsor Cash Flows: IRR {sponsor_irr:.1%}, MOIC {sponsor_moic:.2f}x",
)
fig.update_yaxes(title="EUR millions")
fig.show()


,Metric,Value
0,Exit EBITDA,283.24
1,Exit Enterprise Value,"1,982.66"
2,Exit net debt,"1,047.92"
3,Exit Equity Value,934.73
4,Sponsor equity invested,508.21
5,IRR,0.23
6,MOIC,1.84


,Year,Sponsor cash flow
0,Year 0,-508.21
1,Year 1,0.00
2,Year 2,0.00
3,Year 3,934.73


## 9. Value Creation Bridge

This bridge explains where the sponsor's exit equity value came from.


In [10]:
# Operating improvement is incremental EBITDA valued at the entry multiple.
operating_improvement = (exit_ebitda - historical_ebitda) * entry_multiple

# Multiple effect captures whether the exit multiple is higher or lower than entry.
multiple_effect = exit_ebitda * (exit_multiple - entry_multiple)

# Debt paydown is value transferred from lenders back to the sponsor.
debt_paydown = opening_total_debt - exit_net_debt

# Entry fees reduce sponsor value.
entry_fees = use_fees

value_creation_bridge = pd.DataFrame(
    {
        "Driver": [
            "Sponsor equity invested",
            "Operating improvement",
            "Multiple effect",
            "Debt paydown",
            "Entry fees",
            "Exit equity value",
        ],
        "Value": [
            sponsor_equity,
            operating_improvement,
            multiple_effect,
            debt_paydown,
            -entry_fees,
            exit_equity_value,
        ],
    }
)

display(value_creation_bridge)

fig = px.bar(
    value_creation_bridge,
    x="Driver",
    y="Value",
    text=value_creation_bridge["Value"].map(lambda x: f"{x:,.1f}"),
    title="Value Creation Bridge",
)
fig.update_yaxes(title="EUR millions")
fig.update_xaxes(tickangle=-25)
fig.update_traces(textposition="outside")
fig.show()


,Driver,Value
0,Sponsor equity invested,508.21
1,Operating improvement,297.06
2,Multiple effect,0.00
3,Debt paydown,137.90
4,Entry fees,-8.43
5,Exit equity value,934.73


## 10. Sensitivity Analysis

Only now do we wrap the case in a function. The reason is practical: sensitivity analysis reruns the same model many times.


In [11]:
def run_case_for_sensitivity(entry_multiple_input, exit_multiple_input, ebit_growth_shift_input=0.0, d_and_a_growth_shift_input=0.0):
    # This function uses the same financial logic as the notebook above.
    # It is intentionally used only for repeated sensitivity calculations.

    # Recalculate historical EBITDA.
    case_historical_ebitda = historical_ebit + historical_d_and_a

    # Recalculate entry valuation.
    case_entry_ev = case_historical_ebitda * entry_multiple_input
    case_equity_value = case_entry_ev - net_debt
    case_fees = case_entry_ev * fees_pct_of_ev
    case_total_uses = case_equity_value + net_debt + case_fees

    # Recalculate funding.
    case_senior_debt = case_total_uses * senior_debt_pct
    case_sub_debt = case_total_uses * subordinated_debt_pct
    case_sponsor_equity = case_total_uses - case_senior_debt - case_sub_debt

    # Build the same operating projection, with optional growth shifts.
    case_projection = operating_projection[["Year", "EBIT", "D&A", "OWC", "Capex"]].copy()
    for i in [1, 2, 3]:
        prior_i = i - 1
        case_projection.loc[i, "EBIT"] = case_projection.loc[prior_i, "EBIT"] * (
            case_projection.loc[i, "EBIT"] / operating_projection.loc[prior_i, "EBIT"] + ebit_growth_shift_input
        )
        case_projection.loc[i, "D&A"] = case_projection.loc[prior_i, "D&A"] * (
            case_projection.loc[i, "D&A"] / operating_projection.loc[prior_i, "D&A"] + d_and_a_growth_shift_input
        )
    case_projection["EBITDA"] = case_projection["EBIT"] + case_projection["D&A"]
    case_projection["Change in OWC"] = case_projection["OWC"].shift(1) - case_projection["OWC"]
    case_projection["Change in OWC"] = case_projection["Change in OWC"].fillna(0.0)

    # Build a simplified debt schedule using the same sweep logic.
    senior_balance = case_senior_debt
    sub_balance = case_sub_debt
    cash_balance = 0.0
    case_debt_rows = []

    for year_label in ["Year 1", "Year 2", "Year 3"]:
        row = case_projection.loc[case_projection["Year"] == year_label].iloc[0]
        senior_repay = 0.0
        sub_repay = 0.0

        for _iteration in range(1, 51):
            senior_int = senior_interest_rate * max(senior_balance - 0.5 * senior_repay, 0.0)
            sub_int = subordinated_interest_rate * max(sub_balance - 0.5 * sub_repay, 0.0)
            interest = senior_int + sub_int
            pbt = row["EBIT"] - interest
            tax = pbt * tax_rate
            cash_available = row["EBIT"] + row["D&A"] - interest - tax + row["Change in OWC"] - row["Capex"]
            cash_for_sweep = max(cash_balance + cash_available, 0.0)
            new_senior_repay = min(senior_balance, cash_for_sweep)
            new_sub_repay = min(sub_balance, max(cash_for_sweep - new_senior_repay, 0.0))

            if abs(new_senior_repay - senior_repay) < 1e-10 and abs(new_sub_repay - sub_repay) < 1e-10:
                senior_repay = new_senior_repay
                sub_repay = new_sub_repay
                break

            senior_repay = new_senior_repay
            sub_repay = new_sub_repay

        senior_balance = max(senior_balance - senior_repay, 0.0)
        sub_balance = max(sub_balance - sub_repay, 0.0)
        cash_balance = cash_balance + cash_available - senior_repay - sub_repay
        case_debt_rows.append([year_label, senior_balance, sub_balance, cash_balance])

    case_debt = pd.DataFrame(case_debt_rows, columns=["Year", "Senior debt", "Sub debt", "Cash"])

    # Calculate exit returns.
    case_exit_ebitda = case_projection.loc[case_projection["Year"] == f"Year {exit_year}", "EBITDA"].iloc[0]
    case_exit_ev = case_exit_ebitda * exit_multiple_input
    case_exit_debt_row = case_debt.loc[case_debt["Year"] == f"Year {exit_year}"].iloc[0]
    case_exit_net_debt = case_exit_debt_row["Senior debt"] + case_exit_debt_row["Sub debt"] - case_exit_debt_row["Cash"]
    case_exit_equity = case_exit_ev - case_exit_net_debt
    case_cash_flows = [-case_sponsor_equity] + [0.0] * (exit_year - 1) + [case_exit_equity]
    case_irr = float(npf.irr(case_cash_flows))
    case_moic = case_exit_equity / case_sponsor_equity

    return case_irr, case_moic


# Create grids for the sensitivity tables.
entry_grid = np.arange(6.0, 8.5, 0.5)
exit_grid = np.arange(6.0, 8.5, 0.5)
growth_shift_grid = np.array([-0.02, -0.01, 0.0, 0.01, 0.02])

# Empty tables that will be filled by the loops.
entry_exit_irr = pd.DataFrame(index=entry_grid, columns=exit_grid, dtype=float)
exit_growth_irr = pd.DataFrame(index=growth_shift_grid, columns=exit_grid, dtype=float)
exit_growth_moic = pd.DataFrame(index=growth_shift_grid, columns=exit_grid, dtype=float)

# Sensitivity 1: entry multiple vs exit multiple.
for entry_case in entry_grid:
    for exit_case in exit_grid:
        case_irr, case_moic = run_case_for_sensitivity(entry_case, exit_case)
        entry_exit_irr.loc[entry_case, exit_case] = case_irr

# Sensitivity 2: EBIT growth shift vs exit multiple.
for growth_shift in growth_shift_grid:
    for exit_case in exit_grid:
        case_irr, case_moic = run_case_for_sensitivity(
            entry_multiple,
            exit_case,
            ebit_growth_shift_input=growth_shift,
            d_and_a_growth_shift_input=growth_shift * 0.5,
        )
        exit_growth_irr.loc[growth_shift, exit_case] = case_irr
        exit_growth_moic.loc[growth_shift, exit_case] = case_moic

entry_exit_irr.index.name = "Entry Multiple"
entry_exit_irr.columns.name = "Exit Multiple"
exit_growth_irr.index.name = "EBIT growth shift"
exit_growth_irr.columns.name = "Exit Multiple"
exit_growth_moic.index.name = "EBIT growth shift"
exit_growth_moic.columns.name = "Exit Multiple"

display((entry_exit_irr * 100).round(1).astype(str) + "%")
display(exit_growth_moic.round(2).astype(str) + "x")

# Use text labels for both axes so Plotly treats the heatmap rows and columns as categories.
# This avoids distorted charts when the Y-axis values are small decimals such as -0.02 and 0.02.
entry_axis_labels = [f"{value:.1f}x" for value in entry_exit_irr.index]
exit_axis_labels = [f"{value:.1f}x" for value in entry_exit_irr.columns]
growth_axis_labels = [f"{value:.0%}" for value in exit_growth_irr.index]

fig = px.imshow(
    entry_exit_irr.to_numpy(),
    x=exit_axis_labels,
    y=entry_axis_labels,
    text_auto=".1%",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="IRR Sensitivity: Entry Multiple vs Exit Multiple",
    labels={"x": "Exit Multiple", "y": "Entry Multiple", "color": "IRR"},
)
fig.update_layout(width=850, height=520, margin={"l": 90, "r": 60, "t": 80, "b": 70})
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_xaxes(side="bottom")
fig.show()

fig = px.imshow(
    exit_growth_irr.to_numpy(),
    x=exit_axis_labels,
    y=growth_axis_labels,
    text_auto=".1%",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="IRR Sensitivity: EBIT Growth Shift vs Exit Multiple",
    labels={"x": "Exit Multiple", "y": "EBIT Growth Shift", "color": "IRR"},
)
fig.update_layout(width=850, height=520, margin={"l": 90, "r": 60, "t": 80, "b": 70})
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.update_xaxes(side="bottom")
fig.show()

sensitivities = {
    "entry_exit_irr": entry_exit_irr,
    "exit_growth_irr": exit_growth_irr,
    "exit_growth_moic": exit_growth_moic,
}


Exit Multiple,6.00,6.50,7.00,7.50,8.00
Entry Multiple,,,,,
6.00,24.6%,31.2%,37.2%,42.8%,47.9%
6.50,16.6%,23.5%,29.7%,35.4%,40.7%
7.00,8.6%,16.0%,22.5%,28.4%,33.8%
7.50,0.7%,8.6%,15.5%,21.7%,27.3%
8.00,-7.5%,1.2%,8.6%,15.1%,20.9%


Exit Multiple,6.00,6.50,7.00,7.50,8.00
EBIT growth shift,,,,,
-0.02,1.09x,1.35x,1.62x,1.88x,2.15x
-0.01,1.18x,1.46x,1.73x,2.0x,2.27x
0.00,1.28x,1.56x,1.84x,2.12x,2.4x
0.01,1.38x,1.67x,1.95x,2.24x,2.52x
0.02,1.48x,1.78x,2.07x,2.36x,2.65x


## 11. Reconciliation With The Streamlit Engine

The notebook is written for teaching. The Streamlit app uses `utils.lbo_engine`. This cell checks that both produce the same key outputs.


In [12]:
# Build the nested input dictionary expected by the engine.
BASE_CASE_INPUTS = {
    "deal": {
        "share_price": share_price,
        "diluted_shares": diluted_shares,
        "net_debt": net_debt,
    },
    "valuation": {
        "entry_multiple": entry_multiple,
    },
    "financing": {
        "senior_debt_pct": senior_debt_pct,
        "subordinated_debt_pct": subordinated_debt_pct,
        "senior_interest_rate": senior_interest_rate,
        "subordinated_interest_rate": subordinated_interest_rate,
    },
    "operating": {
        "ebit": historical_ebit,
        "d_and_a": historical_d_and_a,
        "owc": historical_owc,
        "capex": historical_capex,
    },
    "tax_and_fees": {
        "fees_pct_of_ev": fees_pct_of_ev,
        "tax_rate": tax_rate,
    },
    "exit": {
        "exit_multiple": exit_multiple,
        "exit_year": exit_year,
    },
    "projection": {
        "projection_years": 3,
        "ebit_growth_shift": 0.0,
        "d_and_a_growth_shift": 0.0,
        "owc_growth_shift": 0.0,
        "capex_growth_shift": 0.0,
    },
}

# Run the engine with the same base case.
engine_results = run_lbo_model(base_inputs=BASE_CASE_INPUTS)

# Extract engine outputs.
engine_entry = engine_results["entry_valuation"].set_index("Metric")["Value"]
engine_returns = engine_results["returns_summary"].set_index("Metric")["Value"]
engine_credit = engine_results["credit_metrics"]

# Put notebook and engine values next to each other.
reconciliation = pd.DataFrame(
    {
        "Metric": [
            "Entry Enterprise Value",
            "Sponsor equity invested",
            "Exit Equity Value",
            "IRR",
            "MOIC",
            "Year 3 Total Debt / EBITDA",
            "Year 3 EBITDA / Interest",
        ],
        "Notebook": [
            entry_enterprise_value,
            sponsor_equity,
            exit_equity_value,
            sponsor_irr,
            sponsor_moic,
            credit_metrics.loc[credit_metrics["Year"] == "Year 3", "Total Debt / EBITDA"].iloc[0],
            credit_metrics.loc[credit_metrics["Year"] == "Year 3", "EBITDA / Interest expense"].iloc[0],
        ],
        "Engine": [
            engine_entry.loc["Entry Enterprise Value"],
            engine_returns.loc["Sponsor equity invested"],
            engine_returns.loc["Exit Equity Value"],
            engine_returns.loc["IRR"],
            engine_returns.loc["MOIC"],
            engine_credit.loc["Year 3", "Total Debt / EBITDA"],
            engine_credit.loc["Year 3", "EBITDA / Interest expense"],
        ],
    }
)

# Difference should be almost zero.
reconciliation["Difference"] = reconciliation["Notebook"] - reconciliation["Engine"]

# Boolean check for consistency.
reconciliation["Consistent"] = reconciliation["Difference"].abs() < 1e-8

display(reconciliation)

# Stop the notebook if the reconciliation fails.
assert reconciliation["Consistent"].all(), "Notebook and engine do not reconcile."

print("Reconciliation passed.")


,Metric,Notebook,Engine,Difference,Consistent
0,Entry Enterprise Value,"1,685.60","1,685.60",0.00,True
1,Sponsor equity invested,508.21,508.21,0.00,True
2,Exit Equity Value,934.73,934.73,0.00,True
3,IRR,0.23,0.23,0.00,True
4,MOIC,1.84,1.84,0.00,True
5,Year 3 Total Debt / EBITDA,3.70,3.70,0.00,True
6,Year 3 EBITDA / Interest,4.20,4.20,0.00,True


Reconciliation passed.


## 12. Streamlit Demo And Optional Commentary

The app is a productization demo. It is not the core teaching asset.


In [13]:
# Package notebook outputs in the shape expected by the optional reporting layer.
results = {
    "inputs": BASE_CASE_INPUTS,
    "inputs_table": input_table,
    "operating_projection": operating_projection.set_index("Year"),
    "entry_valuation": entry_valuation[["Metric", "Value"]],
    "sources_and_uses": pd.DataFrame(
        {
            "Uses": {
                "Acquisition of equity": use_acquisition_of_equity,
                "Refinanced debt": use_refinanced_debt,
                "Fees": use_fees,
                "Total Uses": total_uses,
            },
            "Sources": {
                "Senior debt": source_senior_debt,
                "Subordinated debt": source_subordinated_debt,
                "Sponsor equity": sponsor_equity,
                "Total Sources": total_sources,
            },
        }
    ),
    "debt_schedule": debt_schedule.set_index("Year"),
    "credit_metrics": credit_metrics.set_index("Year"),
    "returns_summary": returns_summary,
    "value_creation_bridge": value_creation_bridge,
    "sensitivities": sensitivities,
}

streamlit_app_path = PROJECT_ROOT / "app" / "streamlit_lbo_demo.py"

print(f"Streamlit app: {streamlit_app_path}")
print("Run locally with: streamlit run app/streamlit_lbo_demo.py")

# Optional Gemini commentary.
# Keep this False for normal classroom execution so the notebook does not depend on an API key or network access.
RUN_GEMINI_COMMENTARY = False

# Show the payload that would be sent to the reporting layer.
payload = build_commentary_payload(results)
payload_table = pd.DataFrame(payload.items(), columns=["Payload field", "Value"])
display(payload_table)

# If the instructor wants to demo the AI reporting layer, set RUN_GEMINI_COMMENTARY = True and rerun this cell.
if RUN_GEMINI_COMMENTARY:
    commentary = generate_investment_commentary(results)

    if commentary["success"]:
        display(Markdown(commentary["message"]))
    else:
        display(Markdown(f"**Optional commentary unavailable.** {commentary['message']}"))
else:
    display(Markdown("**Gemini commentary not run.** Set `RUN_GEMINI_COMMENTARY = True` in this cell to demo the optional reporting layer."))


Streamlit app: C:\Users\julio\Documents\DemoCode\FinEngineering_AlgoTrading\app\streamlit_lbo_demo.py
Run locally with: streamlit run app/streamlit_lbo_demo.py


,Payload field,Value
0,share_price,2.50
1,implied_offer_price_per_share,4.20
2,takeover_premium,0.68
3,entry_multiple,7.00
4,exit_multiple,7.00
5,exit_year,3.00
6,historical_ebitda,240.80
7,exit_ebitda,283.24
8,ebitda_growth_pct,0.18
9,entry_enterprise_value,"1,685.60"


**Gemini commentary not run.** Set `RUN_GEMINI_COMMENTARY = True` in this cell to demo the optional reporting layer.

## Investment Committee Wrap-Up

- The model starts from operating assumptions, not from a target return.
- Entry valuation determines how much must be funded.
- The financing structure determines interest cost and repayment pressure.
- Operating cash flow repays debt.
- Debt paydown and operating improvement create sponsor value.
- Sensitivity analysis shows whether the base case is robust or fragile.

The Streamlit demo packages the same logic, but the notebook is where the finance should be learned.
